# Step 4 - Data & Evaluation: baseline and first results
Task of the "Data and Evaluation" role (slide 31): **choose the baseline and the comparison metric, run the first evaluation, record results and limitations.**
Same style as `lab4_nim_ml_sklearn.ipynb`: Part 1 by hand with NumPy, Part 2 with scikit-learn. Run cells top to bottom (Shift+Enter).

**Data:** `data/corpus.json`, 12,008 Persian paper abstracts, 7 classes. **Task:** predict `label` from `title + abstract`. The class sizes are very unequal (39.7 percent vs 4.1 percent), so accuracy alone is misleading, like the 12.5 percent losing positions in Nim.

## Cell 1: Load the data and audit it

In [ ]:
import sys, json
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from evaluate_baselines import normalize, metrics_by_hand, summarize, build_models

CORPUS = "../data/corpus.json"
df = pd.DataFrame(json.load(open(CORPUS, encoding="utf-8")))
df["text"] = df["title"].str.strip() + " " + df["abstract"].str.strip()
labels = sorted(df["label"].unique())

print(df.shape)
print(df["label"].value_counts().sort_index())
print("share of the biggest class:", round(df["label"].value_counts(normalize=True).max(), 3))
print("empty abstracts:", (df["abstract"].str.strip() == "").sum(), "  duplicate abstracts:", df.duplicated("abstract").sum())
print("abstract words (min/median/max):", df["abstract"].str.split().str.len().agg(["min", "median", "max"]).tolist())

# Part 1: by hand with NumPy

## Cell 2: Train/test split and the baseline (by hand)

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(df))
n_train = int(0.7 * len(df))
train, test = df.iloc[order[:n_train]], df.iloc[order[n_train:]]
y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()

majority = np.bincount(y_train).argmax()                 # most frequent class in TRAIN
baseline = np.full(len(y_test), majority)                # baseline: always say that class
print("train rows:", len(train), "  test rows:", len(test), "  majority class:", majority)
print("baseline accuracy:", round((baseline == y_test).mean(), 3))

## Cell 3: Metrics by hand (multiclass)

In [ ]:
res = metrics_by_hand(y_test, baseline, labels)
print("confusion matrix (rows = true, columns = predicted):")
print(res["cm"])
print("per-class recall   :", res["recall"].round(3))
print("per-class precision:", res["precision"].round(3))
print(f"accuracy={res['accuracy']:.3f}  macro_recall={res['macro_recall']:.3f}  macro_F1={res['macro_f1']:.3f}")

# Part 2: the same project with scikit-learn

## Cell 4: Baselines and simple models, same metrics

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(df["text"], df["label"], test_size=0.3, random_state=0, stratify=df["label"])

models = build_models()
rows, preds = [], {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    preds[name] = model.predict(X_te)
    rows.append({"model": name, **summarize(y_te, preds[name])})
table1 = pd.DataFrame(rows).round(3)
print(table1.to_string(index=False))

## Cell 5: Is the result stable? (5-fold cross validation)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score, accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv = []
for name in ["B1 majority class", "M2 TF-IDF + LogisticRegression"]:
    f1s, accs = [], []
    for tr, te in skf.split(df["text"], df["label"]):
        m = clone(models[name]).fit(df["text"].iloc[tr], df["label"].iloc[tr])
        p = m.predict(df["text"].iloc[te])
        f1s.append(f1_score(df["label"].iloc[te], p, average="macro", zero_division=0))
        accs.append(accuracy_score(df["label"].iloc[te], p))
    cv.append({"model": name, "acc": np.mean(accs), "macro_F1": np.mean(f1s), "macro_F1_std": np.std(f1s)})
print(pd.DataFrame(cv).round(3).to_string(index=False))

## Cell 6: Per-class report and confusion matrix

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
best = "M2 TF-IDF + LogisticRegression"
print(classification_report(y_te, preds[best], digits=3))
ConfusionMatrixDisplay.from_predictions(y_te, preds[best], cmap="Blues", colorbar=False)
plt.title(best + ", test set")
plt.show()

## Cell 7: Which input field carries the signal? (limitation check)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from evaluate_baselines import tfidf

abl = []
for field in ["title", "abstract", "text"]:
    m = make_pipeline(tfidf(), LinearSVC(C=0.5, random_state=0)).fit(df.loc[X_tr.index, field], y_tr)
    abl.append({"input": field, **summarize(y_te, m.predict(df.loc[X_te.index, field]))})
print(pd.DataFrame(abl).round(3).to_string(index=False))

## Interpretation (observation, reason, conclusion)
| Observation | Reason | Conclusion |
|---|---|---|
| Majority baseline has accuracy 0.397 but macro-F1 0.081 and recall 0 for six of seven classes | Class 5 is 39.7 percent of the data; the baseline never predicts anything else | Accuracy is a trap here; use macro-F1 as the main metric |
| Random baselines: priors 0.233 accuracy / 0.142 macro-F1, uniform 0.143 / 0.125 | No information from the text is used | Any real model must beat macro-F1 0.142 |
| TF-IDF + LogisticRegression: accuracy 0.828, macro-F1 0.785 (5-fold: 0.789 +/- 0.008) | Words and word pairs separate the subject areas | The text carries strong signal; a simple linear model is a solid reference for later steps |
| Class 4 has the lowest recall (0.523), most often mistaken for class 2 (81) and class 3 (40) | Its vocabulary overlaps with neighbouring areas | Report per-class recall, not only the average |
| Title alone gives macro-F1 0.717, abstract alone 0.773, both 0.785 | The abstract has more words; the title adds a little | Keep both fields as input |

## Limitations
See `README.md`, section 7.